# Оптическое умножение: LCD–LCD–камера

Структура примера соответствует исходному репозиторию. Вход и веса
кодируют пропускание по интенсивности. Оптические произведения считываются
камерой; суммирование по строкам пока цифровое.

Ниже используется быстрое идеальное реле. Совпадение с цифровым результатом
проверяет программную основу, а не физическую точность будущего стенда.


In [ ]:
import torch
import source
from dataclasses import asdict

torch.set_num_threads(4)
torch.manual_seed(42)
config = source.Config.load("config.pth")
asdict(config)


## Геометрия и дискретизация
Все координаты — в метрах.


In [ ]:
print(config.planes())
print(config.sampling_report())


## Создание оптической системы

`ideal` — идеальное изображение двумя реле. Опциональный `effective_na`
задаёт приближение эквивалентного зрачка; это не точное описание двух
конечных апертур. Для прямого sinc-расчёта нужна существенно более плотная
сетка и отдельная проверка сходимости (см. Readme.md).


In [ ]:
mul = source.OpticalMul(config, backend="ideal")
x = 0.1 + 0.8 * torch.rand(config.columns, dtype=torch.float64)
W = (0.1 + 0.8 * torch.rand(config.rows, config.columns, dtype=torch.float64)).requires_grad_()
optical = mul(x, W)
digital = W @ x
print("Оптический ответ:", optical.detach())
print("Цифровой ответ:", digital.detach())
print("Относительная ошибка:", ((optical-digital).norm()/digital.norm()).item())


## Интенсивность на камере
Массив относится к расчётной сетке, а не к нативным пикселям сенсора.


In [ ]:
intensity = mul.camera_intensity(x, W)
print("Форма:", tuple(intensity.shape))
print("Диапазон:", intensity.min().item(), intensity.max().item())


## Градиент и проверка конечными разностями


In [ ]:
target = torch.linspace(0.5, 1.5, config.rows, dtype=torch.float64)
loss = (optical-target).square().mean()
gradient, = torch.autograd.grad(loss, W)
direction = torch.randn_like(W)
direction /= direction.norm()
epsilon = 1e-5
with torch.no_grad():
    lp = (mul(x, W+epsilon*direction)-target).square().mean()
    lm = (mul(x, W-epsilon*direction)-target).square().mean()
autograd_value = (gradient*direction).sum().item()
finite_difference = ((lp-lm)/(2*epsilon)).item()
print("Autograd:", autograd_value)
print("Конечная разность:", finite_difference)


## Сохранение конфигурации

Файл содержит только словарь параметров нашей схемы. Модель и веса
сохраняются отдельно, если это потребуется в дальнейшем.


In [ ]:
config.save("config.pth")
assert source.Config.load("config.pth") == config


## Когерентное освещение

Гауссов источник → D → L0 → LCD1 рассчитываются один раз при создании модели.
Пучок и неподвижный фазовый рассеиватель заданы модельными параметрами Config.
Нормировка на цифровой ответ и выравнивание освещения не выполняются.
Интенсивность на LCD1: `mul.illumination_field.abs().square()`.


In [ ]:
# Поле на LCD1 уже рассчитано при создании mul.
print(config.illumination_sampling_report())
print("Поле освещения:", mul.illumination_field.shape)
print("Дополнительный кэш, МиБ:", mul.illumination_field.numel() * mul.illumination_field.element_size() / 2**20)
